In [1]:
import os
from pathlib import Path

In [2]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project'

In [5]:
from dataclasses import dataclass
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [6]:

from WineQuality_Project.constants import *
from WineQuality_Project.utils.common import read_yaml,create_directories

In [7]:
class ConfigManager:
    def __init__(self,
                 config_filepath: Path = CONFIG_FILE_PATH,
                 params_filepath: Path = PARAMS_FILE_PATH,
                 schema_filepath: Path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([Path(self.config['artifact_root'])])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        
        config = self.config['data_ingestion']
        create_directories([Path(config['root_dir'])])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config['root_dir']),
            source_url=config['source_url'],
            local_data_file=Path(config['local_data_file']),
            unzip_dir=Path(config['unzip_dir'])
        )
        return data_ingestion_config

In [8]:
# src/WineQuality_Project/components/data_ingestion.py

import urllib.request
import zipfile
from pathlib import Path
from WineQuality_Project.config import configuration
from WineQuality_Project.utils.common import create_directories
import logging

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        create_directories([self.config.root_dir])

    def download_data(self):
        # local_data_file is a Path object; use its exists() method
        if not self.config.local_data_file.exists():
            filename, headers = urllib.request.urlretrieve(
                url=self.config.source_url,
                filename=str(self.config.local_data_file)
            )
            logging.info(f"File downloaded successfully and saved to {filename} with headers {headers}")
        else:
            logging.info(f"File already exists at {self.config.local_data_file}")

    def extract_zip_file(self):
        """
        Extracts the zip file to the specified directory.
        """
        unzip_path = Path(self.config.unzip_dir)
        unzip_path.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logging.info(f"Extracted zip file to {unzip_path}")

    def initiate_data_ingestion(self):
        self.download_data()
        self.extract_zip_file()
        return self.config.unzip_dir

In [9]:
import logging
from WineQuality_Project.config import configuration

logging.basicConfig(level=logging.INFO, format='[%(asctime)s]: %(message)s')

try:
    config = ConfigManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    csv_folder = data_ingestion.initiate_data_ingestion()
    logging.info(f"CSV files are ready in: {csv_folder}")
except Exception as e:
    logging.error(f"Data ingestion failed: {e}")
    raise e


[2025-12-11 21:25:37] [INFO] WineQualityLogger - YAML file: d:\Data Science\project Series\End_to_end_ReD_Wine_Quality_Mlops_Project\config\config.yml loaded successfully
[2025-12-11 21:25:37,927]: YAML file: d:\Data Science\project Series\End_to_end_ReD_Wine_Quality_Mlops_Project\config\config.yml loaded successfully
[2025-12-11 21:25:37] [INFO] WineQualityLogger - YAML file: d:\Data Science\project Series\End_to_end_ReD_Wine_Quality_Mlops_Project\params.yaml loaded successfully
[2025-12-11 21:25:37,932]: YAML file: d:\Data Science\project Series\End_to_end_ReD_Wine_Quality_Mlops_Project\params.yaml loaded successfully
[2025-12-11 21:25:37] [INFO] WineQualityLogger - YAML file: d:\Data Science\project Series\End_to_end_ReD_Wine_Quality_Mlops_Project\schema.yaml loaded successfully
[2025-12-11 21:25:37,937]: YAML file: d:\Data Science\project Series\End_to_end_ReD_Wine_Quality_Mlops_Project\schema.yaml loaded successfully
[2025-12-11 21:25:37] [INFO] WineQualityLogger - Directory creat